In [0]:
from pyspark.sql import functions as F
df_bronze = spark.read.table("bronze_credit_raw")
total = df_bronze.count()
null_counts = df_bronze.select([
    F.round(F.sum(F.col(c).isNull().cast("int")) / total * 100, 2).alias(c)
    for c in df_bronze.columns
])
display(null_counts)

In [0]:
display(df_bronze.select(
    F.min("age").alias("age_min"), F.max("age").alias("age_max"),
    F.min("credit_score").alias("cs_min"), F.max("credit_score").alias("cs_max"),
    F.min("debt_ratio").alias("dr_min"), F.max("debt_ratio").alias("dr_max"),
    F.min("income").alias("income_min"), F.max("income").alias("income_max")
))

In [0]:
display(df_bronze.groupBy("default").count().orderBy("default"))

In [0]:
df_silver = df_bronze \
    .filter(F.col("age") > 0) \
    .filter(F.col("debt_ratio") >= 0) \
    .withColumn("credit_score",
        F.when(F.col("credit_score") > 850, None)
         .otherwise(F.col("credit_score"))
    ) \
    .dropna(subset=["income", "loan_amount"])

In [0]:
median_cs = df_silver.approxQuantile("credit_score", [0.5], 0.01)[0]
print(f"credit_score 中位數：{median_cs}")

df_silver = df_silver.fillna({"credit_score": median_cs})

null_check = df_silver.select([
    F.sum(F.col(c).isNull().cast("int")).alias(c)
    for c in df_silver.columns
])
display(null_check)

In [0]:
df_silver.write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable("silver_credit")